# PharmaVision Defect Detection — Training v3
**Model:** YOLOv11n | **Dataset:** 1284 train / 161 val | **Full dataset + Augmentation**

In [ ]:
%pip install ultralytics --quiet

## 1 — Setup & Dataset Verification

In [ ]:
from ultralytics import settings
settings.update({'dvc': False, 'mlflow': False})

from ultralytics import YOLO
from pathlib import Path
import yaml

# ── Paths ──────────────────────────────────────────────────────────────────
project_root       = Path(r'C:\Users\Admin\Desktop\Projects\PharmaVision Defect Detection')
dataset_root       = project_root / 'final_dataset.v1i.yolov8'
dataset_yaml       = dataset_root / 'data.yaml'
fixed_dataset_yaml = project_root / 'Training' / 'data.fixed.yaml'
runs_dir           = project_root / 'Training' / 'runs' / 'detect'

# ── Write absolute-path YAML (avoids relative-path issues) ────────────────
with dataset_yaml.open('r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)
data_cfg['path'] = dataset_root.as_posix()
with fixed_dataset_yaml.open('w', encoding='utf-8') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

print(f'Using data config : {fixed_dataset_yaml}')
print(f'Dataset root      : {dataset_root}')

# ── Dataset size check ────────────────────────────────────────────────────
train_imgs = list((dataset_root / 'train' / 'images').glob('*.*'))
valid_imgs = list((dataset_root / 'valid' / 'images').glob('*.*'))
print(f'\nTrain images : {len(train_imgs)}')
print(f'Valid images : {len(valid_imgs)}')
print(f'Classes      : {data_cfg["names"]}')
print('\n✅ All checks passed — training will use the FULL train set.')

## 2 — Train (Full Dataset + Augmentation)

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────
# Using YOLOv11n (latest nano, ~5.6 MB) — better accuracy than v8n
model = YOLO(str(project_root / 'Training' / 'yolo11n.pt'))

results = model.train(
    # ── Data ──────────────────────────────────────────────────────────────
    data     = str(fixed_dataset_yaml),
    fraction = 1.0,          # ← FULL train set (1284 images)
    imgsz    = 640,
    batch    = 8,
    workers  = 2,

    # ── Run identity ──────────────────────────────────────────────────────
    project  = str(runs_dir),
    name     = 'train_v3',
    exist_ok = False,

    # ── Training schedule ─────────────────────────────────────────────────
    epochs        = 150,
    cos_lr        = True,
    lr0           = 0.005,
    lrf           = 0.01,
    warmup_epochs = 5,
    patience      = 30,      # more patience on a larger dataset

    # ── Regularisation ────────────────────────────────────────────────────
    dropout      = 0.2,      # slightly reduced — more data → less dropout needed
    weight_decay = 0.001,

    # ── Geometric augmentation ────────────────────────────────────────────
    degrees     = 15.0,      # rotation ±15°
    translate   = 0.1,
    scale       = 0.5,
    shear       = 3.0,
    perspective = 0.0005,
    fliplr      = 0.5,
    flipud      = 0.05,

    # ── Colour augmentation ───────────────────────────────────────────────
    hsv_h = 0.015,
    hsv_s = 0.7,
    hsv_v = 0.4,
    erasing = 0.3,

    # ── Mosaic / mix augmentation ─────────────────────────────────────────
    mosaic       = 1.0,      # always on
    mixup        = 0.1,
    copy_paste   = 0.1,
    close_mosaic = 20,       # disable mosaic in last 20 epochs for fine-tuning

    # ── Misc ──────────────────────────────────────────────────────────────
    amp   = True,
    plots = True,
)

print(f'\n✅ Training complete!')
print(f'Best weights : {results.save_dir}/weights/best.pt')
print(f'Last weights : {results.save_dir}/weights/last.pt')

## 3 — Validation Metrics

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import numpy as np
import matplotlib.pyplot as plt

project_root       = Path(r'C:\Users\Admin\Desktop\Projects\PharmaVision Defect Detection')
fixed_dataset_yaml = project_root / 'Training' / 'data.fixed.yaml'

# Load best weights (works even if kernel was restarted)
if 'model' not in globals() or 'results' not in globals():
    best_pt = project_root / 'Training' / 'runs' / 'detect' / 'train_v3' / 'weights' / 'best.pt'
    model   = YOLO(str(best_pt))

metrics = model.val(data=str(fixed_dataset_yaml))
print(f'mAP50    : {metrics.box.map50:.4f}')
print(f'mAP50-95 : {metrics.box.map:.4f}')

## 4 — Confusion Matrix

In [ ]:
cm = metrics.confusion_matrix.matrix
names = metrics.names
class_names = [names[i] for i in sorted(names.keys())] if isinstance(names, dict) else list(names)
tick_labels = class_names + ['background']

plt.figure(figsize=(10, 8))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.title('Confusion Matrix — PharmaVision v3', fontsize=14)
plt.colorbar()
plt.xticks(np.arange(len(tick_labels)), tick_labels, rotation=45, ha='right')
plt.yticks(np.arange(len(tick_labels)), tick_labels)
threshold = cm.max() / 2 if cm.size else 0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        v = cm[i, j]
        plt.text(j, i, f'{int(v)}' if float(v).is_integer() else f'{v:.2f}',
                 ha='center', va='center', color='white' if v > threshold else 'black')
plt.ylabel('True class')
plt.xlabel('Predicted class')
plt.tight_layout()
plt.show()

## 5 — Per-Class Performance Table

In [ ]:
import pandas as pd

ordered_class_names = [metrics.names[i] for i in sorted(metrics.names.keys())] \
    if isinstance(metrics.names, dict) else list(metrics.names)

box = metrics.box
rows = [{
    'Class'    : c,
    'Precision': float(box.p[i]),
    'Recall'   : float(box.r[i]),
    'mAP50'    : float(box.ap50[i]),
    'mAP50-95' : float(box.ap[i])
} for i, c in enumerate(ordered_class_names)]

overall = {
    'Class'    : 'Overall',
    'Precision': float(box.mp),
    'Recall'   : float(box.mr),
    'mAP50'    : float(box.map50),
    'mAP50-95' : float(box.map)
}

df = pd.DataFrame([overall] + rows)
df[['Precision','Recall','mAP50','mAP50-95']] = (
    df[['Precision','Recall','mAP50','mAP50-95']] * 100
).round(2)
print('Model Performance Matrix — v3 (%)')
display(df)

## 6 — Live Webcam Detection

In [ ]:
import cv2
from ultralytics import YOLO
from pathlib import Path

project_root = Path(r'C:\Users\Admin\Desktop\Projects\PharmaVision Defect Detection')

if 'model' not in globals():
    best_pt = project_root / 'Training' / 'runs' / 'detect' / 'train_v3' / 'weights' / 'best.pt'
    model   = YOLO(str(best_pt))

cap = cv2.VideoCapture(0)
print("Starting live detection. Press 'q' to stop.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    annotated = model(frame, verbose=False)[0].plot()
    cv2.imshow('PharmaVision v3 – Live Detection', annotated)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()